In [1]:
from fastai.collab import *
from fastai.tabular.all import *
set_seed(42)

In [2]:
import numpy as np
import pandas as pd

Importing the data

In [3]:
path = untar_data(URLs.ML_100k)

<div><progress max="4924029" value="4931584"></progress> 100.15% [4931584/4924029 00:00&lt;00:00]</div>

Reading the data

In [4]:
rating_df = pd.read_csv(path/'u.data',delimiter = '\t', header = None, names = ['user','movie','rating','timestamp'])
rating_df.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Reading the dataframe having a movie title

In [5]:
movies_df = pd.read_csv(path/'u.item', delimiter='|',encoding='latin-1',usecols=(0,1), names=('movie','title'), header=None)  #here movie represents the movieid
movies_df.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


Merging the movies and rating dataframe

In [6]:
rating_df = rating_df.merge(movies_df)  #merging the rating and movie dataframe
rating_df.head()

,user,movie,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


Creation of dataloaders


In [7]:
dls = CollabDataLoaders.from_df(rating_df,item_name = 'title' , bs = 64) #here we are creating a dataloaders based on userID, movie_title as item  and the movie_rating done by the user
dls.show_batch()

,user,title,rating
0,782,Starship Troopers (1997),2
1,943,Judge Dredd (1995),3
2,758,Mission: Impossible (1996),4
3,94,Farewell My Concubine (1993),5
4,23,Psycho (1960),4
5,296,Secrets & Lies (1996),5
6,940,"American President, The (1995)",4
7,334,Star Trek VI: The Undiscovered Country (1991),1
8,380,Braveheart (1995),4
9,690,So I Married an Axe Murderer (1993),1


In [8]:
n_users = len(dls.classes['user']) #number of users
n_movies = len(dls.classes['title']) #number of movies
print('The number of users is:', n_users)
print('The number of movies is:', n_movies)

The number of users is: 944
The number of movies is: 1665


creation of user factor and movie factor

In [9]:
n_factors = 5
user_factors = torch.randn(n_users,n_factors)
movie_factors = torch.randn(n_movies,n_factors)

Collaborative filtering from scratch

In [10]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        return (users * movies).sum(dim = 1)  #prediction of movie rating
        
        

In [11]:
x, y, = dls.one_batch()
y.shape

torch.Size([64, 1])

Creation of model

In [12]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())


Training the model

In [13]:
learn.fit_one_cycle(5, 5e-3) #using 5e-3 as learning rate

epoch,train_loss,valid_loss,time
0,1.323818,1.340935,00:07
1,1.017078,1.092095,00:06
2,0.873056,0.974699,00:06
3,0.762055,0.897630,00:06
4,0.719645,0.874279,00:06


Making the prediction range between 0 and 5

In [14]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors,y_range = (0,5)):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
        self.y_range = y_range
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        return sigmoid_range((users * movies).sum(dim = 1),*self.y_range)  #prediction of movie rating constructed to be within the range of 0 and 5
        
        

In [15]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.955417,0.999712,00:06
1,0.719479,0.925105,00:06
2,0.522617,0.914310,00:06
3,0.423134,0.907511,00:06
4,0.414449,0.905504,00:06


Including the bias for both user and movie

In [16]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors,y_range = (0,5)):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
        self.user_bias = Embedding(n_users,1)
        self.movie_bias = Embedding(n_movies,1)
        self.y_range = y_range
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        rating = (users * movies).sum(dim = 1, keepdim = True)  #prediction of movie rating 
        rating += self.user_bias(x[:,0]) + self.movie_bias(x[:,1])  #addition of both user bias and movie bias
        return sigmoid_range(rating,*self.y_range)  #using sigmoid function to construct the rating of a movie to be within 0 and 5 
        

In [17]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.897461,0.960630,00:07
1,0.632315,0.898508,00:07
2,0.477742,0.905640,00:07
3,0.371390,0.911508,00:07
4,0.357702,0.911395,00:07


As we can observe from the above result that our model is overfitting , cause even though the training error is decreasing, the validation error is increasing. So we must use the regularization to solve this issue.

In [18]:
learn.fit_one_cycle(5,5e-3,wd = 0.1)  #using the weight decay inorder to shrink the value of weight, inorder to prevent overfitting of model

epoch,train_loss,valid_loss,time
0,0.381805,0.911708,00:07
1,0.403445,0.911663,00:07
2,0.381207,0.896106,00:07
3,0.340098,0.883414,00:07
4,0.329895,0.879925,00:07


As we can observe that both the training as well as validation error is decreasing gradually.

Creating our own embedding module

In [19]:
class T(Module):
    def __init__(self):
        self.a = nn.Parameter(torch.ones(3))   #here we are creating a learning parameter for our model
list(T().parameters())       #converting the  pytorch parameters into python list


[Parameter containing:
 tensor([1., 1., 1.], requires_grad=True)]

In [20]:
class T(Module):
    def __init__(self):
        self.a = nn.Linear(1,3,bias = False)
L(T().parameters())        

[Parameter containing:
tensor([[ 0.8222],
        [-0.6335],
        [-0.9844]], requires_grad=True)]

In [21]:
def create_params(size):
    return nn.Parameter(torch.zeros(*size).normal_(0,0.01))  #here nn represents that these parameters are learnable parameters

In [22]:
class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors,y_range = (0,5)):
        self.user_factors = create_params([n_users,n_factors]) #creation of embedding of user factor
        self.movie_factors = create_params([n_movies, n_factors]) #creation of embedding of movie factor
        self.user_bias = create_params([n_users,1])
        self.movie_bias = create_params([n_movies,1])
        self.y_range = y_range
    def forward(self,x):  #here x represents the data pair of user id and movie title
        #as the raw tensor, which we are using in create_params function cannot be called, we must use [] for extracting the required data from user and movie factors
        users = self.user_factors[x[:,0]]  #here the first column represents the userID
        movies = self.movie_factors[x[:,1]]  #the second column represents the movie title
        rating = (users * movies).sum(dim = 1, keepdim = True)  #prediction of movie rating 
        rating += self.user_bias[x[:,0]] + self.movie_bias[x[:,1]]  #addition of both user bias and movie bias
        return sigmoid_range(rating,*self.y_range)  #using sigmoid function to construct the rating of a movie to be within 0 and 5 
        

Training the model but with our custom embedding

In [23]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.919178,0.966418,00:07
1,0.754991,0.890941,00:07
2,0.584899,0.866505,00:07
3,0.533918,0.847862,00:07
4,0.514574,0.842691,00:07


Extracting the movies with lowest bias

In [24]:
movie_bias = learn.model.movie_bias.squeeze()
low_indices = movie_bias.argsort()[:5]  #index of the 5 movies with lowest bias
print('The least popular movies in the list:', [dls.classes['title'][i] for i in low_indices])


The least popular movies in the list: ['Showgirls (1995)', 'Children of the Corn: The Gathering (1996)', 'Lawnmower Man 2: Beyond Cyberspace (1996)', 'Theodore Rex (1995)', 'Grease 2 (1982)']


In [25]:
high_indices = movie_bias.argsort(descending=True)[:5]  #index of the 5 movies with highest bias
print('The most popular movies in the list:', [dls.classes['title'][i] for i in high_indices])

The most popular movies in the list: ['Shawshank Redemption, The (1994)', 'Good Will Hunting (1997)', 'Titanic (1997)', "Schindler's List (1993)", 'As Good As It Gets (1997)']


The rating predicted by the model and the actual rating of the movie

In [26]:
preds, targets = learn.get_preds()

In [27]:
print('The predicted rating of first five movies done by model is:', preds[:5])
print('The actual rating of first five movies is:', targets[:5])


The predicted rating of first five movies done by model is: tensor([[3.8436],
        [2.1517],
        [3.3058],
        [4.1989],
        [4.3911]])
The actual rating of first five movies is: tensor([[4],
        [4],
        [4],
        [5],
        [5]], dtype=torch.int8)


Training the model using fastai.collab

In [28]:
learn = collab_learner(dls, n_factors = 50, y_range = (0,5))
learn.fit_one_cycle(5, 5e-3, wd = 0.1)


epoch,train_loss,valid_loss,time
0,0.863058,0.952211,00:06
1,0.739983,0.890834,00:07
2,0.626048,0.861490,00:07
3,0.528122,0.837625,00:07
4,0.511289,0.833918,00:07


Observing the embedding layers of weight and bias of both users and items(movies)

In [29]:
learn.model

EmbeddingDotBias(
  (u_weight): Embedding(944, 50)
  (i_weight): Embedding(1665, 50)
  (u_bias): Embedding(944, 1)
  (i_bias): Embedding(1665, 1)
)

Extracting the movie bias 

In [30]:
movie_bias = learn.model.i_bias.weight.squeeze()  #as the i_bias is a module, we must use weight inorder to extract the value of bias from tensors of i_bias module
low_indices = movie_bias.argsort()[:5]
print('The movies with lowest bias are:', [dls.classes['title'][i] for i in low_indices])

The movies with lowest bias are: ['Theodore Rex (1995)', 'Children of the Corn: The Gathering (1996)', 'Showgirls (1995)', 'Lawnmower Man 2: Beyond Cyberspace (1996)', 'Grease 2 (1982)']


In [31]:
movie_bias = learn.model.i_bias.weight.squeeze()  #as the i_bias is a module, we must use weight inorder to extract the value of bias from tensors of i_bias module
high_indices = movie_bias.argsort(descending = True)[:5]
print('The movies with highest bias are:', [dls.classes['title'][i] for i in high_indices])

The movies with highest bias are: ['Titanic (1997)', 'L.A. Confidential (1997)', 'Rear Window (1954)', 'Shawshank Redemption, The (1994)', 'Good Will Hunting (1997)']


In [32]:
movie_factors = learn.model.i_weight.weight    #extracting the learned latent taste vector of all the movies
idx = dls.classes['title'].o2i['Titanic (1997)']  #extracting the index of movie Titanic
distance = nn.CosineSimilarity(dim = 1)(movie_factors , movie_factors[idx][None])  #here we are making the second movie factor 2D dimensional inorder to measure the cosine similarity based on this second movie factor
#here the distance represents the cosine similarity value, not the actual distance.
#so higher the value, more close the movies will be.
closest_index = distance.argsort(descending=True)  #as the very first movie is Titanic itself, so we are using the second index
closest_movie = dls.classes['title'][closest_index][1]
second_closest_movie = dls.classes['title'][closest_index][2]
print('The movie closest to Titanic is:', closest_movie)
print('The movie second closest to Titanic is:', second_closest_movie)

The movie closest to Titanic is: 8 Seconds (1994)
The movie second closest to Titanic is: Little Princess, A (1995)


Deep learning for collaborative filtering

In [33]:
embs = get_emb_sz(dls)  #extracting the size of embedding from the dataloaders
embs

[(944, 74), (1665, 102)]